## EDA de Resistencias Antimicrobianas

Primero importamos las librerias que vamos a utilizar

In [1]:
import pandas as pd

Cargamos los dataset. Están en formato .csv separados por ",": Seleccionamos las columnas con las que trabajar y cambiamos los nombres de las columnas. 

In [15]:

# Carga individual de cada archivo en una sola línea
import pandas as pd

cohort = pd.read_csv('./data/cohort.csv', sep=',', usecols=['anon_id', 'pat_enc_csn_id_coded',
                                                             'order_proc_id_coded','ordering_mode', 
                                                             'culture_description','was_positive',
                                                               'organism', 'antibiotic', 'susceptibility'])
demographics = pd.read_csv('./data/demographics.csv', sep=',',usecols=['anon_id', 'pat_enc_csn_id_coded', 
                                                                      'order_proc_id_coded', 'age','gender'])
resistance = pd.read_csv('./data/microbial_resistance.csv', sep=',',  usecols=['anon_id', 'pat_enc_csn_id_coded', 
                                                                               'order_proc_id_coded',
                                                                               'organism', 'antibiotic',
                                                                               'resistant_time_to_culturetime'])
ward = pd.read_csv('./data/ward_info.csv', sep=',', usecols=['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded',
                                                            'hosp_ward_IP', 'hosp_ward_OP',
                                                             'hosp_ward_ER', 'hosp_ward_ICU'])
adi = pd.read_csv('./data/adi_scores.csv', sep=',', usecols=['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded',
                                                              'adi_score', 'adi_state_rank'])


/var/folders/pr/c0xf4wvs7xz313lcynwg4pn80000gp/T/ipykernel_13899/1695826180.py:8: DtypeWarning: Columns (4: gender) have mixed types. Specify dtype option on import or set low_memory=False.
  demographics = pd.read_csv('./data/demographics.csv', sep=',',usecols=['anon_id', 'pat_enc_csn_id_coded',
/var/folders/pr/c0xf4wvs7xz313lcynwg4pn80000gp/T/ipykernel_13899/1695826180.py:17: DtypeWarning: Columns (4: adi_score, 5: adi_state_rank) have mixed types. Specify dtype option on import or set low_memory=False.
  adi = pd.read_csv('./data/adi_scores.csv', sep=',', usecols=['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded',


In [17]:
# Renombrar IDs
for df in [cohort, demographics, resistance, ward, adi]:
    df.rename(columns={
        'order_proc_id_coded': 'culture_id',
        'pat_enc_csn_id_coded': 'episode_id'
    }, inplace=True)

# Renombrar las variables de ward
ward.rename(columns={
    'hosp_ward_IP': 'IP',
    'hosp_ward_OP': 'OP',
    'hosp_ward_ER': 'ER',
    'hosp_ward_ICU': 'ICU'
}, inplace=True)

Decidimos que columnas nos interesan para nuestro EDA:

Todas estan unidas por un ``anon_id``, ``episode_id``y ``culture_id``. Que utilizaremos para unir los diferentes dataframes.

* **Cohort**: ordering_mode, culture_description, was_positive, organism, antibiotic, susceptibility
* **Demographics**: age, gender
* **Resistance**: organism, antibiotic, resistant_time_to_culturetime (days)
* **Ward**: hosp_ward_IP, hosp_ward_OP,hosp_ward_ER, hosp_ward_ICU
* **ADI**: adi_score, adi_state_rank

Definiciones relevantes: 

- **Unidad de análisis:** un cultivo (`culture_id`).
- **Población:** cultivos con `was_positive == True` y al menos un resultado de sensibilidad.
- **AMR:** el cultivo tiene al menos un antibiótico con `susceptibility == Resistant`. Los "Intermediate" se excluyen con la posibilidad de analizarse aparte.
- **`resistant_time_to_culturetime`:** días entre la confirmación de una resistencia previa y la orden del cultivo. Es un **predictor** (historial), nunca el desenlace.